# Sample-Size Sensitivity

This notebook measures how the number of observations affects recovery of the active direction in

$$u=\sin(x+y)+\varepsilon.$$

The analytical direction is $\theta/\pi=0.25$. MI-DL and SI-DL are compared over 15 sample sizes and 60 independent random repeats.

The notebook follows a linear workflow. It contains no helper functions and no `main()` entry point.


## 1. Imports and project paths

Load the local MI-DL and SI-DL implementations together with the numerical and plotting libraries.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.model_selection import train_test_split

OUTPUT_DIR = Path.cwd()
CSV_DIR = OUTPUT_DIR / "csv"
PROJECT_ROOT = OUTPUT_DIR.parents[1]
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))

for project_path in (PROJECT_ROOT / "Compare" / "MI-DL", PROJECT_ROOT / "SI-DL-main"):
    if str(project_path) not in sys.path:
        sys.path.insert(0, str(project_path))

import midl
import SI_DL


## 2. Experiment settings

Each repeat first generates 1,000 observations. Smaller datasets are nested prefixes of that same realization, which isolates the effect of sample size from changes in the underlying random draw.


In [ ]:
BASE_RANDOM_STATE = 42
N_REPEATS = 60
N_TOTAL = 1000
NOISE_LEVEL = 0.10
TEST_SIZE = 0.25

SAMPLE_SIZES = (30, 40, 50, 60, 75, 100, 130, 160, 200, 300, 400, 500, 650, 800, 1000)
PLOT_XTICKS = (30, 50, 75, 100, 200, 300, 500, 800, 1000)
TRUE_THETA = np.pi / 4.0
TRUE_THETA_OVER_PI = 0.25

THETA_BOUNDS = (0.0, np.pi)
DE_MAXITER = 20
DE_POPSIZE = 8
DE_TOL = 0.001
DE_POLISH = True
MI_K_NEIGHBORS = 10

SI_BANDWIDTH_GRID = np.linspace(0.03, 0.09, 13)
SI_CV_FOLDS = 5
BANDWIDTH_PILOT_REPEATS = 20

BANDWIDTHS_CSV = CSV_DIR / "bandwidths.csv"
RESULTS_CSV = CSV_DIR / "results.csv"
SUMMARY_CSV = CSV_DIR / "summary.csv"
PLOT_PATH = OUTPUT_DIR / "sample_size.png"

METHOD_COLORS = {"MI-DL": "#3268a8", "SI-DL": "#d26b4f"}
CSV_DIR.mkdir(parents=True, exist_ok=True)


## 3. Select one SI-DL bandwidth per sample size

Bandwidth selection is separated from direction optimization. For the first 20 random realizations, five-fold cross-validation is performed at the known reference direction $\theta=\pi/4$. The median selected bandwidth is then fixed for the full comparison.

This makes the experiment self-contained without repeatedly performing bandwidth cross-validation inside every optimization step.


In [ ]:
bandwidth_records = []

for pilot_index in range(BANDWIDTH_PILOT_REPEATS):
    random_state = BASE_RANDOM_STATE + pilot_index
    rng = np.random.default_rng(random_state)
    x_full = rng.uniform(0.0, 5.0, size=N_TOTAL)
    y_full = rng.uniform(0.0, 5.0, size=N_TOTAL)
    u_clean = np.sin(x_full + y_full)
    noise_scale = NOISE_LEVEL * np.std(u_clean, ddof=0)
    u_full = u_clean + rng.normal(0.0, noise_scale, size=N_TOTAL)

    for n_samples in SAMPLE_SIZES:
        train_idx, _ = train_test_split(
            np.arange(n_samples),
            test_size=TEST_SIZE,
            random_state=random_state,
        )
        reference_coordinate = (
            np.cos(TRUE_THETA) * x_full[:n_samples]
            + np.sin(TRUE_THETA) * y_full[:n_samples]
        )
        score = SI_DL.cross_validated_explained_variance_score(
            reference_coordinate[train_idx],
            u_full[:n_samples][train_idx],
            bandwidths=SI_BANDWIDTH_GRID,
            cv=SI_CV_FOLDS,
            random_state=random_state,
        )
        bandwidth_records.append({
            "pilot_repeat": pilot_index,
            "n_samples": n_samples,
            "selected_bandwidth": float(score["bandwidth"]),
        })

    print(f"Bandwidth pilot {pilot_index + 1}/{BANDWIDTH_PILOT_REPEATS}")

bandwidth_trials = pd.DataFrame(bandwidth_records)
bandwidths = (
    bandwidth_trials.groupby("n_samples", as_index=False)["selected_bandwidth"]
    .median()
    .rename(columns={"selected_bandwidth": "bandwidth"})
)
bandwidths.to_csv(BANDWIDTHS_CSV, index=False)
fixed_bandwidth = dict(zip(bandwidths["n_samples"], bandwidths["bandwidth"]))
bandwidths



## 4. Run the code

In [ ]:
result_records = []

for repeat_index in range(N_REPEATS):
    random_state = BASE_RANDOM_STATE + repeat_index
    rng = np.random.default_rng(random_state)
    x_full = rng.uniform(0.0, 5.0, size=N_TOTAL)
    y_full = rng.uniform(0.0, 5.0, size=N_TOTAL)
    u_clean = np.sin(x_full + y_full)
    noise_scale = NOISE_LEVEL * np.std(u_clean, ddof=0)
    u_full = u_clean + rng.normal(0.0, noise_scale, size=N_TOTAL)

    for n_samples in SAMPLE_SIZES:
        x = x_full[:n_samples]
        y = y_full[:n_samples]
        u = u_full[:n_samples]
        train_idx, _ = train_test_split(
            np.arange(n_samples),
            test_size=TEST_SIZE,
            random_state=random_state,
        )
        x_train = x[train_idx]
        y_train = y[train_idx]
        u_train = u[train_idx]
        error_scale = float(np.var(u_train, ddof=0))

        mi_result = differential_evolution(
            lambda parameters: float(
                midl.information_lower_bound(
                    np.cos(float(parameters[0])) * x_train
                    + np.sin(float(parameters[0])) * y_train,
                    u_train,
                    k_neighbors=MI_K_NEIGHBORS,
                    random_state=random_state,
                )["epsilon_lb"]
                / error_scale
            ),
            bounds=[THETA_BOUNDS],
            maxiter=DE_MAXITER,
            popsize=DE_POPSIZE,
            tol=DE_TOL,
            seed=random_state + n_samples * 101,
            polish=DE_POLISH,
            updating="immediate",
            workers=1,
        )
        mi_theta = float(mi_result.x[0])
        mi_coordinate = np.cos(mi_theta) * x_train + np.sin(mi_theta) * y_train
        mi_values = midl.information_lower_bound(
            mi_coordinate,
            u_train,
            k_neighbors=MI_K_NEIGHBORS,
            random_state=random_state,
        )
        mi_objective = float(mi_values["epsilon_lb"] / error_scale)
        mi_theta_over_pi = mi_theta / np.pi
        mi_abs_error = abs(mi_theta_over_pi - TRUE_THETA_OVER_PI)
        result_records.append({
            "repeat": repeat_index,
            "random_state": random_state,
            "method": "MI-DL",
            "n_samples": n_samples,
            "theta_over_pi": mi_theta_over_pi,
            "theta_abs_error": mi_abs_error,
            "normalized_error": mi_abs_error / TRUE_THETA_OVER_PI,
            "objective_error": mi_objective,
            "mutual_information": float(mi_values["mutual_information"]),
            "si_score": np.nan,
            "bandwidth": np.nan,
        })

        bandwidth = float(fixed_bandwidth[n_samples])
        si_result = differential_evolution(
            lambda parameters: float(
                1.0
                - SI_DL.explained_variance_score(
                    np.cos(float(parameters[0])) * x_train
                    + np.sin(float(parameters[0])) * y_train,
                    u_train,
                    bandwidth=bandwidth,
                )["S_cov"]
            ),
            bounds=[THETA_BOUNDS],
            maxiter=DE_MAXITER,
            popsize=DE_POPSIZE,
            tol=DE_TOL,
            seed=random_state + n_samples * 101 + 10_000,
            polish=DE_POLISH,
            updating="immediate",
            workers=1,
        )
        si_theta = float(si_result.x[0])
        si_coordinate = np.cos(si_theta) * x_train + np.sin(si_theta) * y_train
        si_values = SI_DL.explained_variance_score(
            si_coordinate,
            u_train,
            bandwidth=bandwidth,
        )
        si_objective = float(1.0 - si_values["S_cov"])
        si_theta_over_pi = si_theta / np.pi
        si_abs_error = abs(si_theta_over_pi - TRUE_THETA_OVER_PI)
        result_records.append({
            "repeat": repeat_index,
            "random_state": random_state,
            "method": "SI-DL",
            "n_samples": n_samples,
            "theta_over_pi": si_theta_over_pi,
            "theta_abs_error": si_abs_error,
            "normalized_error": si_abs_error / TRUE_THETA_OVER_PI,
            "objective_error": si_objective,
            "mutual_information": np.nan,
            "si_score": float(si_values["S_cov"]),
            "bandwidth": bandwidth,
        })

    print(f"Experiment repeat {repeat_index + 1}/{N_REPEATS}")

results = pd.DataFrame(result_records)
results.to_csv(RESULTS_CSV, index=False)
print(f"Saved {len(results)} rows to {RESULTS_CSV.relative_to(OUTPUT_DIR)}")

summary_records = []

for (method, n_samples), group in results.groupby(["method", "n_samples"], sort=False):
    error_values = group["normalized_error"].to_numpy(float)
    objective_values = group["objective_error"].to_numpy(float)
    summary_records.append({
        "method": method,
        "n_samples": int(n_samples),
        "n_repeats": int(group["repeat"].nunique()),
        "error_q25": float(np.quantile(error_values, 0.25)),
        "error_median": float(np.median(error_values)),
        "error_q75": float(np.quantile(error_values, 0.75)),
        "objective_median": float(np.median(objective_values)),
        "theta_median": float(group["theta_over_pi"].median()),
    })

summary = pd.DataFrame(summary_records)
summary.to_csv(SUMMARY_CSV, index=False)
summary


## 5. Plot the sample-size trend

Only one figure is retained. The lines show the median normalized direction error, and the shaded regions show the 25th–75th percentile range across 60 repeats.


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.8))
positive_values = summary.loc[summary["error_q25"] > 0.0, "error_q25"]
plot_floor = float(positive_values.min() * 0.2) if not positive_values.empty else 1.0e-8

for method, group in summary.groupby("method", sort=False):
    group = group.sort_values("n_samples")
    x_values = group["n_samples"].to_numpy(float)
    median = np.maximum(group["error_median"].to_numpy(float), plot_floor)
    lower = np.maximum(group["error_q25"].to_numpy(float), plot_floor)
    upper = np.maximum(group["error_q75"].to_numpy(float), plot_floor * 1.01)
    ax.plot(
        x_values,
        median,
        marker="o",
        markersize=6.5,
        linewidth=2.4,
        color=METHOD_COLORS[method],
        label=method,
    )
    ax.fill_between(
        x_values,
        lower,
        upper,
        color=METHOD_COLORS[method],
        alpha=0.18,
        label=f"{method} 25%-75%",
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(min(SAMPLE_SIZES) * 0.92, max(SAMPLE_SIZES) * 1.08)
ax.set_xticks(PLOT_XTICKS)
ax.set_xticklabels([str(value) for value in PLOT_XTICKS])
ax.set_xlabel("Number of Data Points", fontsize=15)
ax.set_ylabel("Normalized Direction Error", fontsize=15)
ax.tick_params(axis="both", which="major", labelsize=13)
ax.tick_params(axis="both", which="minor", labelsize=11)
ax.legend(frameon=False, fontsize=11)
fig.tight_layout()
fig.savefig(PLOT_PATH, dpi=240)
plt.show()

print(f"Saved {PLOT_PATH.name}")

pd.DataFrame({
    "file": [
        str(BANDWIDTHS_CSV.relative_to(OUTPUT_DIR)),
        str(RESULTS_CSV.relative_to(OUTPUT_DIR)),
        str(SUMMARY_CSV.relative_to(OUTPUT_DIR)),
        PLOT_PATH.name,
    ],
    "description": [
        "Fixed SI-DL bandwidth by sample size",
        "All optimized directions from 60 repeats",
        "Median and interquartile summaries",
        "Final sample-size comparison",
    ],
})
